# Solucion: Clasificacion Binaria de Radiografias de Torax

Este notebook construye una CNN para clasificar radiografias de torax en dos clases: `NORMAL` y `PNEUMONIA`.

La actividad se desarrolla en dos fases:

1. Modelo base sin `data augmentation`
2. Modelo mejorado con `data augmentation`

La estructura sigue la metodologia del notebook `perrosygatos.ipynb`, pero adaptada al problema medico actual.

## Consideraciones iniciales

- Se debe usar la particion existente del dataset: `train`, `val` y `test`.
- Se debe ignorar la carpeta duplicada `chest_xray/chest_xray`.
- Se debe ignorar la carpeta `__MACOSX`.
- La resolucion no se reducira en exceso para conservar detalles pulmonares relevantes.
- Para este notebook se trabajara con `180x180`, evitando tamanos demasiado bajos como `64x64` o `96x96`.

In [ ]:
# Imports y configuracion base.
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8')
sns.set_palette('deep')

IMG_SIZE = (180, 180)
BATCH_SIZE = 32
EPOCHS = 8

In [ ]:
# Rutas oficiales del dataset.
BASE_DIR = Path('chest_xray')
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR = BASE_DIR / 'val'
TEST_DIR = BASE_DIR / 'test'

for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not split_dir.exists():
        raise FileNotFoundError(f'No se encontro la carpeta esperada: {split_dir.resolve()}')

print('Dataset base:', BASE_DIR.resolve())
print('Train:', TRAIN_DIR.resolve())
print('Val:', VAL_DIR.resolve())
print('Test:', TEST_DIR.resolve())

## 1. Revision rapida del dataset

Primero contamos las imagenes por clase para confirmar la estructura del dataset y observar el desbalance entre `NORMAL` y `PNEUMONIA`.

In [ ]:
# Conteo de imagenes por split y clase.
def count_images_by_split(base_dir):
    records = []
    for split in ['train', 'val', 'test']:
        split_path = base_dir / split
        for class_name in sorted([d.name for d in split_path.iterdir() if d.is_dir()]):
            total = len([f for f in (split_path / class_name).iterdir() if f.is_file()])
            records.append({'split': split, 'class_name': class_name, 'images': total})
    return pd.DataFrame(records)

counts_df = count_images_by_split(BASE_DIR)
counts_df

In [ ]:
# Grafico de distribucion de imagenes.
plt.figure(figsize=(9, 5))
sns.barplot(data=counts_df, x='split', y='images', hue='class_name')
plt.title('Distribucion de imagenes por split y clase')
plt.xlabel('Split')
plt.ylabel('Numero de imagenes')
plt.show()

## 2. Generadores sin data augmentation

En la fase base solo se aplica reescalado para normalizar los pixeles entre 0 y 1.

In [ ]:
# Generadores de la fase base.
train_gen = ImageDataGenerator(rescale=1.0 / 255)
val_gen = ImageDataGenerator(rescale=1.0 / 255)
test_gen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED,
)

validation_generator = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED,
)

test_generator = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

class_indices = train_generator.class_indices
idx_to_class = {value: key for key, value in class_indices.items()}

print('Mapeo de clases:', class_indices)

In [ ]:
# Funcion para visualizar muestras del generador.
def plot_samples(generator, n_images=8, title='Muestras del generador'):
    images, labels = next(generator)
    n_images = min(n_images, len(images))

    plt.figure(figsize=(14, 7))
    for i in range(n_images):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].squeeze(), cmap='gray')
        label_name = idx_to_class[int(labels[i])]
        plt.title(label_name)
        plt.axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_samples(train_generator, title='Muestras del conjunto de entrenamiento')

## 3. Modelo CNN base

Se usa una CNN sencilla y estable, cercana al estilo del notebook de referencia, pero adaptada a imagenes medicas en escala de grises.

In [ ]:
# Funcion para construir la CNN.
def build_cnn_model(input_shape=(180, 180, 1)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.30),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

base_model = build_cnn_model(input_shape=IMG_SIZE + (1,))
base_model.summary()

In [ ]:
# Compilacion del modelo base.
base_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Entrenamiento del modelo base.
base_history = base_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    verbose=1,
)

## 4. Evaluacion del modelo base

Ahora evaluamos el rendimiento en `test` y revisamos la matriz de confusion y el reporte por clase.

In [ ]:
# Evaluacion base en el conjunto de prueba.
base_test_loss, base_test_acc = base_model.evaluate(test_generator, verbose=1)
print(f'Loss en test: {base_test_loss:.4f}')
print(f'Accuracy en test: {base_test_acc:.4f}')

In [ ]:
# Predicciones y metricas detalladas.
def evaluate_binary_model(model, generator, model_name='Modelo'):
    generator.reset()
    probs = model.predict(generator, verbose=1).ravel()
    preds = (probs >= 0.5).astype(int)
    y_true = generator.classes

    cm = confusion_matrix(y_true, preds)
    report = classification_report(y_true, preds, target_names=[idx_to_class[0], idx_to_class[1]], output_dict=True)
    report_df = pd.DataFrame(report).transpose()

    print(f'Reporte de clasificacion - {model_name}')
    display(report_df)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[idx_to_class[0], idx_to_class[1]],
                yticklabels=[idx_to_class[0], idx_to_class[1]])
    plt.title(f'Matriz de confusion - {model_name}')
    plt.xlabel('Prediccion')
    plt.ylabel('Valor real')
    plt.show()

    return probs, preds, y_true, report_df

base_probs, base_preds, base_true, base_report_df = evaluate_binary_model(base_model, test_generator, 'Modelo base')

In [ ]:
# Curvas de entrenamiento.
def plot_history(history, title_suffix='Modelo'):
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history_df['accuracy'], label='train_accuracy')
    axes[0].plot(history_df['val_accuracy'], label='val_accuracy')
    axes[0].set_title(f'Accuracy - {title_suffix}')
    axes[0].set_xlabel('Epocas')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history_df['loss'], label='train_loss')
    axes[1].plot(history_df['val_loss'], label='val_loss')
    axes[1].set_title(f'Loss - {title_suffix}')
    axes[1].set_xlabel('Epocas')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(base_history, 'Modelo base')

In [ ]:
# Funcion para ver predicciones correctas e incorrectas.
def show_predictions(model, generator, probs, preds, y_true, correct=True, n_images=6, title='Predicciones'):
    filenames = np.array(generator.filenames)
    selected_idx = np.where(preds == y_true)[0] if correct else np.where(preds != y_true)[0]

    if len(selected_idx) == 0:
        print('No hay ejemplos disponibles para esta vista.')
        return

    selected_idx = selected_idx[:n_images]
    plt.figure(figsize=(14, 8))

    for i, idx in enumerate(selected_idx, start=1):
        img_path = str(generator.directory) + '/' + str(filenames[idx])
        image = tf.keras.utils.load_img(img_path, color_mode='grayscale', target_size=IMG_SIZE)
        image = tf.keras.utils.img_to_array(image)

        plt.subplot(2, 3, i)
        plt.imshow(image.squeeze(), cmap='gray')
        real_label = idx_to_class[int(y_true[idx])]
        pred_label = idx_to_class[int(preds[idx])]
        prob = probs[idx]
        title_text = f'Real: {real_label} | Pred: {pred_label} | Prob: {prob:.3f}'
        plt.title(title_text)
        plt.axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_predictions(base_model, test_generator, base_probs, base_preds, base_true, correct=True, title='Predicciones correctas - Modelo base')
show_predictions(base_model, test_generator, base_probs, base_preds, base_true, correct=False, title='Predicciones incorrectas - Modelo base')

## 5. Segunda fase: modelo con data augmentation

En esta etapa solo se modifica el generador de entrenamiento. La arquitectura del modelo se mantiene igual para que la comparacion sea justa.

In [ ]:
# Generador con augmentation moderado.
train_gen_aug = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.05,
    height_shift_range=0.05,
)

validation_gen_aug = ImageDataGenerator(rescale=1.0 / 255)
test_gen_aug = ImageDataGenerator(rescale=1.0 / 255)

train_generator_aug = train_gen_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED,
)

validation_generator_aug = validation_gen_aug.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED,
)

test_generator_aug = test_gen_aug.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

In [ ]:
# Entrenamiento del modelo con augmentation.
aug_model = build_cnn_model(input_shape=IMG_SIZE + (1,))
aug_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

aug_history = aug_model.fit(
    train_generator_aug,
    epochs=EPOCHS,
    validation_data=validation_generator_aug,
    verbose=1,
)

In [ ]:
# Evaluacion del modelo con augmentation.
aug_test_loss, aug_test_acc = aug_model.evaluate(test_generator_aug, verbose=1)
print(f'Loss en test: {aug_test_loss:.4f}')
print(f'Accuracy en test: {aug_test_acc:.4f}')

aug_probs, aug_preds, aug_true, aug_report_df = evaluate_binary_model(aug_model, test_generator_aug, 'Modelo con augmentation')

In [ ]:
# Curvas del modelo con augmentation.
plot_history(aug_history, 'Modelo con augmentation')

In [ ]:
# Tabla comparativa final.
comparison_df = pd.DataFrame([
    {
        'modelo': 'base',
        'test_loss': base_test_loss,
        'test_accuracy': base_test_acc,
        'precision_pneumonia': base_report_df.loc['PNEUMONIA', 'precision'],
        'recall_pneumonia': base_report_df.loc['PNEUMONIA', 'recall'],
        'f1_pneumonia': base_report_df.loc['PNEUMONIA', 'f1-score'],
    },
    {
        'modelo': 'augmentation',
        'test_loss': aug_test_loss,
        'test_accuracy': aug_test_acc,
        'precision_pneumonia': aug_report_df.loc['PNEUMONIA', 'precision'],
        'recall_pneumonia': aug_report_df.loc['PNEUMONIA', 'recall'],
        'f1_pneumonia': aug_report_df.loc['PNEUMONIA', 'f1-score'],
    }
])

comparison_df

In [ ]:
# Comparacion visual de accuracy y loss.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(base_history.history['val_accuracy'], label='base_val_accuracy')
axes[0].plot(aug_history.history['val_accuracy'], label='aug_val_accuracy')
axes[0].set_title('Comparacion de val_accuracy')
axes[0].set_xlabel('Epocas')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(base_history.history['val_loss'], label='base_val_loss')
axes[1].plot(aug_history.history['val_loss'], label='aug_val_loss')
axes[1].set_title('Comparacion de val_loss')
axes[1].set_xlabel('Epocas')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ejemplos del modelo con augmentation.
show_predictions(aug_model, test_generator_aug, aug_probs, aug_preds, aug_true, correct=True, title='Predicciones correctas - Modelo con augmentation')
show_predictions(aug_model, test_generator_aug, aug_probs, aug_preds, aug_true, correct=False, title='Predicciones incorrectas - Modelo con augmentation')

## 6. Conclusiones

Al finalizar el entrenamiento, completa esta seccion con una interpretacion breve:

- si el modelo base presento overfitting o no
- si el augmentation mejoro la generalizacion
- si el recall de `PNEUMONIA` mejoro o empeoro
- cual de los dos modelos conviene reportar como resultado final

En este problema es importante observar no solo `accuracy`, sino tambien el comportamiento sobre la clase `PNEUMONIA`.